SI disclosure: OpenAI Codex assisted with code, analysis and documentation. Source satellite measurements are observed data. This shared copy has saved execution outputs removed. See PROVENANCE.json and SOURCES_Jakarta.md.

# Analog Atlas — a fresh Earth search with OSEMN

**Goal:** find overlooked Earth patches worth testing for a specific Moon/Mars operation.

**Result to read first after running:** the **N — iNterpret** table. It shows **where, terrain match, prior research and next test**.

| OSEMN | One job |
|---|---|
| **Obtain** | Search 384 fresh Earth patches; preview each dataset with `head()` |
| **Scrub** | Exclude cataloged analog areas and keep strict coverage masks |
| **Explore** | See where the measured patches are and what is missing |
| **Model** | Compare regional terrain with Moon south-pole and Mars Jezero patches |
| **iNterpret** | Check research; explain each lead; extract minerals, temperature and snow |

**Patagonia is already a published Mars-analog region.** It is excluded here, together with other cited sites. The old 96-location pilot is not used as the fresh result.

**Kaggle:** enable Internet and CPU. Add and enable `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` in Secrets for satellite downloads. The terrain search and literature API need no keys. Shared maps use OpenStreetMap; no CARTO key is loaded.

This is a terrain-led discovery screen, not a global census or proof of a new analog. Earth's atmosphere, gravity and radiation differ. A useful analogy is tied to a test, not to permanent-base suitability.

## O — Obtain

Change only these settings to start. **384 patches = 4× the old sample.** Sampling spans occupied latitude/longitude bands within ±80°, subject to available DEM tiles. Prior pilot locations are skipped. Allow several GB for terrain and **up to 2 GB per satellite candidate**; CPU and network speed control runtime.

`RUN_FRESH_SEARCH=False` only reloads an existing matching fresh run. Missing or failed data never falls back to the old pilot.

In [ ]:
RUN_FRESH_SEARCH = True
SAMPLES = 384
SEED = 20261114
WORKERS = 4
KNOWN_SITE_BUFFER_KM = 100
EXCLUDE_PREVIOUS_PILOT = True
SHORTLIST_PER_TARGET = 5
RUN_RESEARCH_CHECK = True
REFRESH_RESEARCH = False  # True to refresh cached Crossref metadata
SATELLITE_CANDIDATE_COUNT = 3
TARGET = 'mars_jezero'  # Or 'moon_south_pole'
GRID_M = '1000'        # Recheck with '2000'
MIN_OPEN_FRACTION = 0.5
WEIGHTS = dict(slope_p25_deg=.10,slope_median_deg=.20,slope_p90_deg=.15,
               relief_m=.25,roughness_m=.20,low_slope_fraction=.10)
# Add local geological names/alternate spellings after the first research screen.
PLACE_ALIASES = {}  # {'candidate_id': ['local basin name', 'regional name']}
# Human review must include sources and an explanation of geographic overlap.
MANUAL_REVIEW = {}  # See the example in N; no record is declared new automatically.

In [ ]:
# Runtime setup; no GPU required. Package versions are saved in the run manifest.
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
    'numpy>=1.26,<3','pandas>=2.2,<4','matplotlib>=3.9,<4',
    'rasterio>=1.3.10,<2','pyproj>=3.6,<4','shapely>=2,<3','folium>=0.17,<1','requests>=2.31,<3'])

In [ ]:
# Bundled readable helpers are also included in the ZIP.
import base64,zlib,json,math,hashlib,sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
BASE=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
HELPER_DIR=BASE/'analog_atlas_helpers';HELPER_DIR.mkdir(exist_ok=True)
BUNDLE=json.loads(zlib.decompress(base64.b64decode('...')))
for name,source in BUNDLE.items():(HELPER_DIR/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(HELPER_DIR))
import importlib
import discovery_search as search
import novelty_review as review
import candidate_visuals
importlib.reload(search);importlib.reload(review);importlib.reload(candidate_visuals)
FEATURES=search.FEATURES
CACHE=BASE/'analog_atlas_cache';CACHE.mkdir(exist_ok=True)
FRESH_DIR=BASE/f'discovery_{SAMPLES}_seed{SEED}';FRESH_DIR.mkdir(exist_ok=True)
EXPORT_DIR=BASE/'analog_atlas_exports';EXPORT_DIR.mkdir(exist_ok=True)
def show_head(name,frame):
    print(f'{name}: {frame.shape[0]} rows x {frame.shape[1]} columns')
    display(frame.head())

In [ ]:
DATASET_CATALOG = [('Copernicus DEM GLO-90', 'Earth', 'Terrain features for fresh sampled patches', 'USED in terrain ranking', 'https://registry.opendata.aws/copernicus-dem/'), ('LRO LOLA LDEM_75S_240M', 'Moon', 'South-pole terrain target', 'USED in terrain ranking', 'https://pds-geosciences.wustl.edu/missions/lro/lola.htm'), ('MGS MOLA MEGT44N000HB', 'Mars', 'Jezero terrain target', 'USED in terrain ranking', 'https://pds-geosciences.wustl.edu/missions/mgs/megdr.html'), ('ESA WorldCover 2021 v200', 'Earth', 'Bare/sparse vegetation and snow/ice screening', 'USED for surface filter', 'https://esa-worldcover.org/en/data-access'), ('NASA POWER 2001–2020', 'Earth', 'Annual air temperature, precipitation, humidity', 'Optional context; not downloaded by this fresh workflow', 'https://power.larc.nasa.gov/docs/services/api/temporal/climatology/'), ('EMITL2BMIN V001', 'Earth', 'Mineral IDs, band depth, fit and uncertainty', 'Downloaded for shortlisted candidates when Earthdata Secrets are enabled', 'https://doi.org/10.5067/EMIT/EMITL2BMIN.001'), ('EMITL2ARFL V001: MASK companion only', 'Earth', 'Cloud/cirrus/water/spacecraft quality flags', 'USED to mask EMIT detections; reflectance cube not downloaded', 'https://doi.org/10.5067/EMIT/EMITL2ARFL.001'), ('MOD11A1 V061', 'Earth', 'Daily day/night surface temperature and QC', 'Downloaded for shortlisted candidates when Earthdata Secrets are enabled', 'https://doi.org/10.5067/MODIS/MOD11A1.061'), ('MOD10A1 V6.1; CMR version 61', 'Earth', 'Daily snow index and QA', 'Downloaded for shortlisted candidates when Earthdata Secrets are enabled', 'https://doi.org/10.5067/MODIS/MOD10A1.061'), ('ESA CPA + USGS analog inventory', 'Earth', 'Exclude known sites', 'Cited snapshot + live USGS refresh', 'https://doi.org/10.5066/P9J72921'), ('Natural Earth admin-1 v5.1.2', 'Earth', 'Regional place names', 'Search labels only', 'https://www.naturalearthdata.com/downloads/10m-cultural-vectors/10m-admin-1-states-provinces/'), ('Crossref scholarly metadata', 'Literature', 'Screen prior analog research', 'Title/available abstract search; full papers need review', 'https://www.crossref.org/documentation/retrieve-metadata/rest-api/')]
datasets_df=pd.DataFrame(DATASET_CATALOG,columns=['dataset','body','purpose','use','link'])
display(datasets_df)
KNOWN=json.loads(BUNDLE['known_analogs.json'])
if RUN_FRESH_SEARCH:KNOWN=review.refresh_usgs(KNOWN,CACHE/'research')
review.dump(FRESH_DIR/'known_analogs_input.json',KNOWN)
known_sites_df=pd.DataFrame(KNOWN['sites'])
show_head('known_sites_df',known_sites_df[['name','kind','lat','lon','source_url']])
print(KNOWN['limits'])
print('USGS refresh:',KNOWN.get('usgs_refresh',{'status':'bundled snapshot'}))

**Exclusion rule:** catalog points get a 100 km buffer; broad known regions use conservative rectangles. Both are expanded by 17 km to cover the entire 24 km buffered DEM patch. These are editable screening choices, not surveyed research boundaries. The registry is incomplete; passing this filter does not prove novelty.

Sources: [ESA catalog](https://esamultimedia.esa.int/docs/gsp/The_Catalogue_of_Planetary_Analogues.pdf), [USGS catalog](https://doi.org/10.5066/P9J72921), [Patagonia research](https://doi.org/10.1016/j.pss.2008.11.006). Each registry row links its source.

In [ ]:
# Obtain and quality-screen measurements only. Ranking happens after the previews.
if RUN_FRESH_SEARCH:
    command=[sys.executable,str(HELPER_DIR/'discovery_search.py'),'--samples',str(SAMPLES),
        '--seed',str(SEED),'--workers',str(WORKERS),'--cache',str(CACHE),'--out',str(FRESH_DIR),
        '--known-sites',str(FRESH_DIR/'known_analogs_input.json'),'--exclusion-km',str(KNOWN_SITE_BUFFER_KM),'--stage','obtain']
    if EXCLUDE_PREVIOUS_PILOT:command+=['--exclude-ids',str(HELPER_DIR/'previous_pilot_ids.json')]
    subprocess.check_call(command)
required=['obtained_candidates.json','target_features.json','obtain_manifest.json','sampled_locations.json','known_analogs_used.json']
if any(not (FRESH_DIR/f).exists() for f in required):
    raise RuntimeError('No complete fresh dataset. Enable RUN_FRESH_SEARCH and rerun Obtain; the old pilot is not substituted.')
DATA={'rows':json.loads((FRESH_DIR/'obtained_candidates.json').read_text()),
      'targets':json.loads((FRESH_DIR/'target_features.json').read_text()),
      'manifest':json.loads((FRESH_DIR/'obtain_manifest.json').read_text()),'names':{}}
OBTAIN_KNOWN=json.loads((FRESH_DIR/'known_analogs_used.json').read_text())
# The current registry may add sites discovered during research review.
# Retain both versions: Scrub applies current + acquisition-time exclusions.
seen_names={s['name'] for s in KNOWN['sites']}
KNOWN['sites'] += [s for s in OBTAIN_KNOWN['sites'] if s['name'] not in seen_names]
if DATA['manifest']['seed']!=SEED or DATA['manifest']['sample_count']!=SAMPLES:
    raise RuntimeError('Saved dataset does not match the requested seed/sample count.')
if DATA['manifest']['exclusion_radius_km']!=KNOWN_SITE_BUFFER_KM or bool(DATA['manifest']['previous_pilot_ids_excluded'])!=EXCLUDE_PREVIOUS_PILOT:
    raise RuntimeError('Saved exclusion settings differ. Run a fresh Obtain stage.')
print('Fresh run:',DATA['manifest']['finished_utc'],'| measured terrain:',DATA['manifest']['valid_terrain_count'])

### Data first: actual `head()` previews

These are measurements derived from downloaded rasters, not mock rows. Terrain is summarized on the chosen grid; the source links and run manifest retain provenance. `head()` shows the first rows, not the best locations. EMIT/MODIS previews appear before their charts in N, after those files are obtained.

In [ ]:
sampled_df=pd.DataFrame(json.loads((FRESH_DIR/'sampled_locations.json').read_text())['candidates'])
terrain_df=pd.DataFrame([{'candidate_id':r['id'],'latitude':r['lat'],'longitude':r['lon'],
    **r['features'][GRID_M],'source_url':r['source_url']} for r in DATA['rows'] if r.get('terrain_status')=='measured'])
planetary_df=pd.DataFrame([{'target':t['id'],'latitude':t['lat'],'longitude':t['lon'],
    **t['features'][GRID_M],'source_url':t['source_url']} for t in DATA['targets']])
landcover_df=pd.DataFrame([{'candidate_id':r['id'],**r.get('landcover',{})} for r in DATA['rows']])
for name,frame in [('sampled_df',sampled_df),('terrain_df',terrain_df),('planetary_df',planetary_df),('landcover_df',landcover_df)]:
    show_head(name,frame)

## S — Scrub

Keep **≥95% valid terrain**, **≥95% valid land-cover support** and **≥50% bare/sparse or snow/ice cover**. Failed coverage stays missing. Known-site exclusions apply again before ranking, including any registry edits made in this session. No mask is relaxed to produce more candidates.

In [ ]:
scrub=[]
for r in DATA['rows']:
    hits=review.exclusion_hits(r['lat'],r['lon'],KNOWN,KNOWN_SITE_BUFFER_KM)
    cover=r.get('landcover',{})
    reason=('known analog area' if hits else 'terrain unavailable' if r.get('terrain_status')!='measured' else
            'land cover unavailable' if cover.get('status')!='sampled' or cover.get('valid_fraction',0)<.95 else
            'insufficient open surface' if cover.get('bare_or_snow_fraction',0)<MIN_OPEN_FRACTION else 'eligible')
    r['discovery_eligible']=reason=='eligible'
    scrub.append({'candidate_id':r['id'],'status':reason,'known_sites':hits})
scrub_df=pd.DataFrame(scrub)
display(scrub_df['status'].value_counts().rename_axis('status').reset_index(name='patches'))
review.dump(FRESH_DIR/'scrub_audit.json',scrub)
review.dump(FRESH_DIR/'known_analogs_after_review.json',KNOWN)
if not any(r['discovery_eligible'] for r in DATA['rows']):
    raise RuntimeError('No eligible patches. Increase SAMPLES/change SEED or inspect failed coverage; keep QA intact.')

## E — Explore

The map shows the search footprint before any similarity ranking. Orange patches pass the screen; gray patches need data or fail the surface filter. High latitude coverage depends on source availability.

In [ ]:
import folium
coverage_map=folium.Map(location=[15,10],zoom_start=2,tiles='OpenStreetMap')
for r in DATA['rows']:
    folium.CircleMarker([r['lat'],r['lon']],radius=3,
        color='#b45309' if r['discovery_eligible'] else '#64748b',fill=True,
        tooltip=f"{r['lat']}, {r['lon']} | {'eligible' if r['discovery_eligible'] else 'excluded / missing'}").add_to(coverage_map)
coverage_map.save(str(EXPORT_DIR/'search_coverage.html'));display(coverage_map)
display(terrain_df[['slope_median_deg','relief_m','roughness_m']].describe().round(2))

## M — Model

**Compare terrain, not habitability.** Both worlds use 20 km patches on 1 km and 2 km grids. The metric is `100 × exp(−weighted normalized differences)`; smaller differences give higher scores. Scales come from this fresh sample, so scores are not directly comparable to the old pilot. Planetary spectra, radiation, thermal cycles and extractable ice are not part of this score.

Targets: Moon **89.5°S, 0°** and Mars Jezero **18.4°N, 77.5°E**. Neither is a certified future permanent base. Change weights in O and rerun M onward.

In [ ]:
search.WEIGHTS=np.array([WEIGHTS[f] for f in FEATURES],dtype=float)
if not np.all(np.isfinite(search.WEIGHTS)) or (search.WEIGHTS<0).any() or search.WEIGHTS.sum()<=0:
    raise ValueError('Weights must be finite, nonnegative and not all zero.')
DATA['calibration']=search.rank(DATA['rows'],DATA['targets'],SEED)
review.dump(FRESH_DIR/'calibration.json',DATA['calibration'])
def rank_candidates(target_id, grid_m, weights, require_open=True, minimum_open=.5):
    grid_m = str(grid_m)
    if not 0 <= minimum_open <= 1:
        raise ValueError('Open-surface fraction must be between 0 and 1.')
    target = next(t for t in DATA['targets'] if t['id'] == target_id)
    config = DATA['calibration'][grid_m]
    w = [float(weights[f]) * int(config['active'][i]) for i,f in enumerate(FEATURES)]
    if any(not math.isfinite(v) or v < 0 for v in w) or sum(w) <= 0:
        raise ValueError('Use finite nonnegative weights, with at least one active weight above zero.')
    w = [v / sum(w) for v in w]
    found = []
    for row in DATA['rows']:
        if row['terrain_status'] != 'measured' or not row.get('discovery_eligible'):
            continue
        cover = row.get('landcover', {})
        fraction = cover.get('bare_or_snow_fraction') if cover.get('status') == 'sampled' else None
        if require_open and (fraction is None or fraction < minimum_open):
            continue
        values = row['features'][grid_m]
        contributions = {f: w[i] * abs(values[f] - target['features'][grid_m][f]) / config['scales'][i]
                         for i,f in enumerate(FEATURES)}
        found.append({
            'id':row['id'], 'region':DATA['names'].get(row['id'],'Unlabelled candidate'),
            'lat':row['lat'], 'lon':row['lon'],
            'terrain_index':100 * math.exp(-sum(contributions.values())),
            'open_fraction':fraction, 'weighted_differences':contributions,
            'measurements':values, 'source_url':row['source_url'],
        })
    return sorted(found,key=lambda r:-r['terrain_index'])

def show_ranking(result, target_id, grid_m):
    import pandas as pd
    if not result:
        print('No eligible candidates. Relax the optional filter or collect the missing evidence.')
        return
    summary = [{k:r[k] for k in ['region','lat','lon','terrain_index','open_fraction']} for r in result[:10]]
    display(pd.DataFrame(summary).round(3))
    target = next(t for t in DATA['targets'] if t['id'] == target_id)
    first = result[0]
    explanation = [{'feature':f,'Earth':first['measurements'][f],
                    'target':target['features'][str(grid_m)][f],
                    'weighted_difference':first['weighted_differences'][f]} for f in FEATURES]
    print(f"Leading patch: {first['lat']}, {first['lon']}; grid {grid_m} m. Lower differences are better.")
    display(pd.DataFrame(explanation).round(3))
    print('Terrain only. Mineralogy, ice, caves, soil mechanics, access and novelty remain unvalidated.')

RANKINGS={t['id']:rank_candidates(t['id'],GRID_M,WEIGHTS,True,MIN_OPEN_FRACTION) for t in DATA['targets']}
SHORTLIST_IDS=list(dict.fromkeys(r['id'] for ranked in RANKINGS.values() for r in ranked[:SHORTLIST_PER_TARGET]))
SHORTLIST=[r for r in DATA['rows'] if r['id'] in SHORTLIST_IDS]
RANKING=RANKINGS[TARGET]
display(pd.DataFrame([{'target':t,'latitude':r['lat'],'longitude':r['lon'],
    'terrain_index':round(r['terrain_index'],1),'candidate_id':r['id']} for t,rs in RANKINGS.items() for r in rs[:SHORTLIST_PER_TARGET]]))
review.dump(FRESH_DIR/'ranked_candidates.json',DATA['rows'])
review.dump(EXPORT_DIR/'terrain_rankings.json',{'grid_m':GRID_M,'weights':WEIGHTS,'rankings':RANKINGS})

### Check the shortlisted places against research

Regional names come from **Natural Earth**, then **Crossref** searches published titles and available abstracts. Read the paper links: a regional mention does not prove the patch was studied. Add basin, volcano, desert or local names to `PLACE_ALIASES` and rerun. Country-wide searches add context but do not by themselves mark a patch known.

| Label | Meaning |
|---|---|
| `POTENTIAL_PRIOR_WORK` | A place name and analog/planet terms occur in metadata; inspect the paper's study area |
| `BROADER_REGION_PRIOR_WORK` | Country-level research exists; exact patch overlap remains unreviewed |
| `PRIOR_REGIONAL_RESEARCH_FOUND` | A supplemental primary source identifies regional analog research missed by the metadata API |
| `NO_MATCH_IN_SEARCHED_METADATA` | No match in these limited searches; **not proof of novelty** |
| `SEARCH_INCOMPLETE` / `PLACE_NAME_NEEDED` | Fix failed queries or supply geographic aliases |
| `KNOWN_SITE` | A cited human review confirms overlap; remove it from discovery leads |

The queries, timestamps, DOI links and failures are saved. No automatic status says “new discovery”.

In [ ]:
if RUN_RESEARCH_CHECK:
    RESEARCH=review.screen_literature(SHORTLIST,CACHE/'research',EXPORT_DIR,PLACE_ALIASES,refresh=REFRESH_RESEARCH)
elif (EXPORT_DIR/'literature_screen.json').exists():
    RESEARCH=json.loads((EXPORT_DIR/'literature_screen.json').read_text())
    if set(r['candidate_id'] for r in RESEARCH['reviews'])!=set(SHORTLIST_IDS):
        raise RuntimeError('Saved research belongs to another shortlist. Enable RUN_RESEARCH_CHECK.')
else:
    RESEARCH={'reviews':[{'candidate_id':r['id'],'place':f"{r['lat']}, {r['lon']}",
        'status':'NOT_SCREENED','decision':'HOLD_FOR_REVIEW','novelty_confirmed':False} for r in SHORTLIST],
        'papers':[],'queries':[],'labels':{}}
DATA['names']={cid:label['label'] for cid,label in RESEARCH.get('labels',{}).items()}
papers_df=pd.DataFrame(RESEARCH['papers'])
queries_df=pd.DataFrame(RESEARCH['queries'])
show_head('papers_df',papers_df)
show_head('queries_df',queries_df)
if not papers_df.empty:
    display(papers_df.sort_values('potential_prior_analog',ascending=False)[
        ['candidate_id','title','year','url','potential_prior_analog']].head(30))
REVIEWS=review.apply_manual_reviews(RESEARCH,MANUAL_REVIEW)
review.dump(EXPORT_DIR/'publication_reviews.json',REVIEWS)

## N — iNterpret

**These are leads for regional mobility/traverse rehearsal.** Terrain resemblance alone does not establish similar minerals, temperatures, buried ice, caves or construction performance.

For a paper you have read, add a record in `MANUAL_REVIEW` and rerun the research/result cells:

```python
MANUAL_REVIEW = {'candidate_id': {
    'decision': 'KNOWN_SITE',  # or PRIOR_WORK_NEARBY / NO_OVERLAP_FOUND / HOLD_FOR_REVIEW
    'sources': ['https://doi.org/...'],
    'geographic_reason': 'Paper study coordinates overlap this 20 km patch.',
    'reviewer': 'Your name', 'reviewed_utc': 'YYYY-MM-DD'
}}
```

`NO_OVERLAP_FOUND` means only that the reviewed papers did not overlap. A novelty claim still needs a wider literature review. Excluded leads are retained in the audit.

In [ ]:
review_by_id={r['candidate_id']:r for r in REVIEWS}
KNOWN_FROM_PAPERS={r['candidate_id'] for r in REVIEWS if r.get('decision')=='KNOWN_SITE'}
RESULT_ROWS=[]
for planet,ranked in RANKINGS.items():
    for r in ranked[:SHORTLIST_PER_TARGET]:
        v=review_by_id.get(r['id'],{})
        RESULT_ROWS.append({'place':v.get('place',f"{r['lat']}, {r['lon']}"),
            'coordinates':f"{r['lat']:.2f}, {r['lon']:.2f}",'target':planet,
            'terrain /100':round(r['terrain_index'],1),'research':v.get('status','NOT_SCREENED'),
            'decision':v.get('decision','HOLD_FOR_REVIEW'),
            'could_test':'Regional rover route planning',
            'next_test':'Read study footprints; then local hazards and traction', 'candidate_id':r['id']})
results_df=pd.DataFrame(RESULT_ROWS)
display(results_df.drop(columns=['candidate_id']))
DISCOVERY_IDS=[cid for cid in SHORTLIST_IDS if cid not in KNOWN_FROM_PAPERS]
RANKING=[r for r in RANKINGS[TARGET] if r['id'] in DISCOVERY_IDS]
review.dump(EXPORT_DIR/'plain_results.json',RESULT_ROWS)
print(f'{len(SHORTLIST_IDS)} unique shortlisted patches; {len(KNOWN_FROM_PAPERS)} excluded after human publication review.')
print('Confirmed new analogs: 0. Permanent-base suitability: not established.')
display(review.simple_map([r for ranked in RANKINGS.values() for r in ranked[:SHORTLIST_PER_TARGET]],REVIEWS,
    EXPORT_DIR/'research_review_map.html'))

### Deeper evidence for the shortlist: repeat Obtain → Scrub → Explore

Download actual EMIT/MODIS measurements for up to three shortlisted leads, including a lower-latitude lead when available, then show their `head()` previews **before charts or evidence checks**. Actual EMIT coverage is determined by the catalog, not assumed. Protected downloads require Earthdata Secrets; missing credentials are reported. This is a follow-up to the terrain search, not a substitute for planetary mineral/thermal measurements.

Temperature coverage still expands from 12 toward 60 dates until the goal of 24 accepted summaries across 6 calendar months or the budget is reached. **All existing masks stay unchanged.** The coverage report explains every rejection. Snow index is not buried-ice volume; EMIT library detections are not bulk mineral abundance.

In [ ]:
EVIDENCE_ENGINE = '"""Six evidence screens. Catalog hits never become scientific validation."""\nimport json\nimport math\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom urllib.parse import urlencode\nfrom urllib.request import urlopen, Request\n\nCMR = \'https://cmr.earthdata.nasa.gov/search/granules.json\'\nSTAC = \'https://planetarycomputer.microsoft.com/api/stac/v1/search\'\nSOURCES = {\n \'emit\': {\'title\':\'EMIT L2BMIN V001\', \'url\':\'https://doi.org/10.5067/EMIT/EMITL2BMIN.001\', \'api\':\'cmr\',\'short_name\':\'EMITL2BMIN\',\'version\':\'001\'},\n \'lst\': {\'title\':\'MOD11A1 daily surface temperature V061\', \'url\':\'https://doi.org/10.5067/MODIS/MOD11A1.061\',\'api\':\'cmr\',\'short_name\':\'MOD11A1\',\'version\':\'061\'},\n \'snow\': {\'title\':\'MOD10A1 daily snow cover V6.1\', \'url\':\'https://doi.org/10.5067/MODIS/MOD10A1.061\',\'api\':\'cmr\',\'short_name\':\'MOD10A1\',\'version\':\'61\'},\n \'dem\': {\'title\':\'Copernicus GLO-30\', \'url\':\'https://planetarycomputer.microsoft.com/dataset/cop-dem-glo-30\',\'api\':\'stac\',\'collection\':\'cop-dem-glo-30\'},\n \'optical\': {\'title\':\'Sentinel-2 L2A\', \'url\':\'https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a\',\'api\':\'stac\',\'collection\':\'sentinel-2-l2a\'},\n}\nCHECKS = {\n \'minerals\': {\'sources\':[\'emit\'], \'satellite\':\'Quality-screened surface mineral detections; compatible planetary observations can support spectral similarity.\', \'field\':\'Sample spectroscopy/XRD, grain size, coatings and bulk composition.\', \'planetary\':\'https://pds-geosciences.wustl.edu/missions/mro/crism.htm\', \'extra\':\'https://pds.nasa.gov/ds-view/pds/viewProfile.jsp?dsid=CH1-ORB-L-M3-4-L2-REFLECTANCE-V1.0\'},\n \'thermal\': {\'sources\':[\'lst\'], \'satellite\':\'Clear-sky surface temperature at observation times; observed range only.\', \'field\':\'Continuous ground logging, heating/cooling rates and equipment cycling under relevant atmosphere/vacuum.\', \'planetary\':\'https://www.diviner.ucla.edu/data\', \'extra\':\'https://modis-land.gsfc.nasa.gov/pdf/MOD11_User_Guide_V61.pdf\'},\n \'ice\': {\'sources\':[\'snow\'], \'satellite\':\'Surface snow cover; model-assisted permafrost context. Neither establishes extractable subsurface ice.\', \'field\':\'Cores, radar, ice fraction/depth, host material and extraction yield.\', \'planetary\':\'https://swim.psi.edu/SWIM4MIMProducts.php\', \'extra\':\'https://climate.esa.int/en/projects/permafrost/\'},\n \'caves\': {\'sources\':[\'optical\',\'dem\'], \'satellite\':\'Regional geology and candidate surface openings if imagery resolves them; many entrances are smaller than Sentinel pixels.\', \'field\':\'Surveyed passages, roof stability, temperature and communication/localization tests.\', \'planetary\':\'https://astrogeology.usgs.gov/search/map/mars_global_cave_candidate_catalog_v1_cushing\', \'extra\':\'https://www.usgs.gov/centers/astrogeology-science-center/science/caves\'},\n \'mobility\': {\'sources\':[\'dem\',\'optical\'], \'satellite\':\'Slopes and obstacles larger than the effective resolution; coarse route-planning context.\', \'field\':\'Wheel-scale terrain, traction, slip, sinkage, energy and localization trials.\', \'planetary\':\'https://pds-geosciences.wustl.edu/missions/mgs/megdr.html\', \'extra\':\'https://pds-geosciences.wustl.edu/missions/lro/lola.htm\'},\n \'construction\': {\'sources\':[\'emit\',\'dem\'], \'satellite\':\'Surface mineral and terrain screening; possible feedstock leads only.\', \'field\':\'Bearing strength, compaction, excavation force, bulk chemistry, processing yield and energy.\', \'planetary\':\'https://pds-geosciences.wustl.edu/missions/mro/crism.htm\', \'extra\':\'https://doi.org/10.5066/F7RR1WDJ\'},\n}\n\ndef query_catalog(source, bbox, start, end, limit=3):\n    """One bounded metadata request; no data downloads or login."""\n    spec=SOURCES[source]\n    if len(bbox)!=4 or not (-180<=bbox[0]<bbox[2]<=180 and -90<=bbox[1]<bbox[3]<=90):\n        raise ValueError(\'Use west,south,east,north; split boxes crossing the date line.\')\n    params={\'bounding_box\':\',\'.join(map(str,bbox)), \'short_name\':spec.get(\'short_name\'),\n            \'version\':spec.get(\'version\'), \'temporal\':f\'{start}T00:00:00Z,{end}T23:59:59Z\',\'page_size\':limit}\n    if spec[\'api\']==\'stac\':\n        params={\'collections\':spec[\'collection\'],\'bbox\':\',\'.join(map(str,bbox)),\'limit\':limit}\n        if source!=\'dem\':params[\'datetime\']=f\'{start}T00:00:00Z/{end}T23:59:59Z\'\n    url=(CMR if spec[\'api\']==\'cmr\' else STAC)+\'?\'+urlencode(params)\n    out={\'query_url\':url,\'checked_utc\':datetime.now(timezone.utc).isoformat(),\'source\':source,\n         \'status\':\'not_queried\',\'interpretation\':\'Catalog overlap only; no valid pixels or physical property confirmed.\'}\n    try:\n        with urlopen(Request(url,headers={\'User-Agent\':\'AnalogAtlas/1.0\'}),timeout=30) as response:\n            data=json.load(response)\n            entries=data.get(\'feed\',{}).get(\'entry\',[]) if spec[\'api\']==\'cmr\' else data.get(\'features\',[])\n            total=response.headers.get(\'CMR-Hits\') if spec[\'api\']==\'cmr\' else data.get(\'numberMatched\')\n        out.update(status=\'catalog_hits\' if entries else \'no_catalog_hits\',total_hits=int(total) if total is not None else None,\n                   returned_count=len(entries),items=entries)\n    except Exception as error:\n        out.update(status=\'request_failed\',error_type=type(error).__name__)\n    return out\n\ndef reviewed_values(record, candidate_id):\n    """Require provenance, candidate scope and explicit QA before interpreting values."""\n    return (isinstance(record,dict) and record.get(\'candidate_id\')==candidate_id\n            and (record.get(\'qa_reviewed\') is True or (record.get(\'qa_processed\') is True and record.get(\'processing\')==\'satellite_extract_v1\')) and bool(record.get(\'source_urls\'))\n            and bool(record.get(\'method\')) and bool(record.get(\'period\')))\n\ndef finite(values):\n    return isinstance(values,list) and len(values)>0 and all(isinstance(x,(int,float)) and not isinstance(x,bool) and math.isfinite(x) for x in values)\n\ndef assess(check, candidate, observation, settings, catalogs):\n    spec=CHECKS[check]\n    result={\'check\':check,\'status\':\'MISSING_MEASUREMENTS\',\'finding\':\'No reviewed observations supplied.\',\n            \'satellite_can_support\':spec[\'satellite\'],\'field_testing_required\':spec[\'field\'],\n            \'operational_validation\':\'NOT_ESTABLISHED\',\n            \'catalog_status\':{s:catalogs.get(s,{}).get(\'status\',\'not_queried\') for s in spec[\'sources\']}}\n    if check==\'mobility\':\n        f=candidate.get(\'features\',{}).get(str(settings[\'terrain_grid_m\']))\n        if f:\n            result.update(status=\'COARSE_TERRAIN_MEASURED\',finding=f"Median slope {f[\'slope_median_deg\']:.2f} degrees; P90 {f[\'slope_p90_deg\']:.2f} degrees at {settings[\'terrain_grid_m\']} m grid. Wheel-scale hazards unresolved.")\n    if not reviewed_values(observation,candidate[\'id\']):\n        if observation.get(\'candidate_id\')==candidate[\'id\'] and observation.get(\'extraction_status\'):\n            state=observation[\'extraction_status\']\n            explanations={\'EARTHDATA_LOGIN_REQUIRED\':\'Add and enable EARTHDATA_USERNAME and EARTHDATA_PASSWORD in Kaggle Secrets, then rerun extraction.\',\n                          \'EARTHDATA_LOGIN_FAILED\':\'Earthdata authentication failed; verify secrets, Internet and account/application authorization.\',\n                          \'NO_GRANULES\':\'No archive granules matched this patch and date range.\',\n                          \'NO_ACCEPTED_MEASUREMENTS\':\'Downloaded observations produced no measurements passing the configured coverage and QA screens.\',\n                          \'EXTRACTION_FAILED\':\'Extraction could not finish. Inspect satellite/extraction_manifest.json for the failing stage and HTTP status.\'}\n            result.update(status=state,finding=explanations.get(state,\'No accepted automatic measurements; inspect the extraction manifest.\'))\n            return result\n        if observation:result[\'input_note\']=\'Input ignored: candidate ID, source URLs, period, method and QA confirmation required.\'\n        return result\n    result[\'provenance\']={k:observation[k] for k in [\'candidate_id\',\'source_urls\',\'period\',\'method\',\'qa_reviewed\']}\n    automatic=observation.get(\'qa_processed\') is True and observation.get(\'processing\')==\'satellite_extract_v1\'\n    result[\'provenance\'][\'quality_check\']=\'automatic product QA; human review pending\' if automatic else \'user-reviewed\'\n    if check in (\'minerals\',\'construction\'):\n        earth=observation.get(\'earth_minerals\',[]); target=observation.get(\'target_minerals\',[])\n        if earth and all(isinstance(x,str) for x in earth):\n            result.update(status=\'SURFACE_DETECTIONS_REVIEWED\',finding=\'Reported Earth detections: \'+\', \'.join(sorted(set(earth))))\n            if automatic:result[\'status\']=\'AUTO_QA_SURFACE_DETECTIONS\'\n            if check==\'minerals\' and target and observation.get(\'cross_sensor_reviewed\') is True:\n                shared=sorted(set(earth)&set(target))\n                result[\'finding\']+=\'; shared reviewed labels: \'+(\', \'.join(shared) or \'none\')+\'. Label overlap is not bulk-composition similarity.\'\n    elif check==\'thermal\':\n        values=observation.get(\'surface_temperature_c\',[])\n        if finite(values):\n            count=len(values); required=settings[\'required_temperature_range_c\']\n            sample_unit=observation.get(\'sample_unit\',\'surface samples\')\n            observation_count=observation.get(\'observation_count\',count)\n            observed=[min(values),max(values)]\n            result.update(status=\'OBSERVED_TEMPERATURE_SAMPLES\',finding=f\'{count} {sample_unit}, {observed[0]:.1f} to {observed[1]:.1f} C; {observation_count} temporal observation summaries. Overpass/cloud gaps remain; no absolute extremes established.\',\n                          observed_range_c=observed,required_range_c=required,\n                          observed_range_spans_requirement=observed[0]<=required[0] and observed[1]>=required[1],\n                          sample_count_screen=observation_count>=settings[\'minimum_thermal_samples\'])\n    elif check==\'ice\':\n        fractions=observation.get(\'snow_cover_fraction\',[])\n        if finite(fractions) and all(0<=x<=1 for x in fractions):\n            result.update(status=\'SURFACE_SNOW_CONTEXT\',finding=f\'Mean valid observed snow fraction: {sum(fractions)/len(fractions):.3f}. Snow is not an extractable subsurface-ice measurement.\')\n            if automatic:result[\'finding\']=f"Mean fraction of QA-valid observed pixels above NDSI threshold {observation[\'snow_ndsi_threshold\']}: {sum(fractions)/len(fractions):.3f}. This is an index-based screen, not subpixel snow fraction or subsurface ice."\n    elif check==\'caves\':\n        count=observation.get(\'reviewed_entrance_candidates\')\n        if isinstance(count,int) and not isinstance(count,bool) and count>=0:\n            result.update(status=\'SURFACE_OPENINGS_REVIEWED\',finding=f\'{count} candidate openings reviewed. Zero means none identified in these observations, not absence of caves. Interiors unverified.\')\n    elif check==\'mobility\':\n        slopes=observation.get(\'slope_deg\',[]); resolution=observation.get(\'grid_m\')\n        if finite(slopes) and all(0<=x<=90 for x in slopes) and isinstance(resolution,(int,float)) and resolution>0:\n            fraction=sum(x<=settings[\'maximum_route_slope_deg\'] for x in slopes)/len(slopes)\n            result.update(status=\'REVIEWED_TERRAIN_SCREEN\',finding=f\'{fraction:.1%} of supplied terrain cells below slope screen at {resolution} m grid; no traction/driveability validation.\')\n    return result\n\ndef template(candidate_id):\n    fields={\'minerals\':{\'earth_minerals\':[],\'target_minerals\':[],\'cross_sensor_reviewed\':False},\n            \'thermal\':{\'surface_temperature_c\':[]},\'ice\':{\'snow_cover_fraction\':[]},\n            \'caves\':{\'reviewed_entrance_candidates\':None},\'mobility\':{\'slope_deg\':[],\'grid_m\':None},\n            \'construction\':{\'earth_minerals\':[]}}\n    return {k:dict(candidate_id=candidate_id,source_urls=[],period=\'\',method=\'\',qa_reviewed=False,**v) for k,v in fields.items()}\n'
exec(compile(EVIDENCE_ENGINE, "evidence_checks.py", "exec"))

In [ ]:
# Choose any measured candidate ID, or keep the highest-ranked patch.
if not DISCOVERY_IDS:raise RuntimeError('All shortlisted candidates are known sites. Increase the sample or shortlist; no replacement is invented.')
EVIDENCE_CANDIDATE_ID = RANKING[0]['id'] if RANKING else DISCOVERY_IDS[0]
EVIDENCE_CANDIDATE = next(r for r in DATA['rows'] if r['id']==EVIDENCE_CANDIDATE_ID)
polygon = EVIDENCE_CANDIDATE['footprint']
EVIDENCE_BBOX = [min(p[0] for p in polygon),min(p[1] for p in polygon),
                 max(p[0] for p in polygon),max(p[1] for p in polygon)]
EVIDENCE_START = '2023-01-01'
EVIDENCE_END = '2023-12-31'
RUN_EVIDENCE_APIS = False  # True: five public metadata requests, no large downloads
SETTINGS = {
    'terrain_grid_m': int(GRID_M),
    'maximum_route_slope_deg': 10,       # Screening choice, not a robot safety limit
    'required_temperature_range_c': [-20,40], # Example test requirement; replace it
    'minimum_thermal_samples': 30,      # Convenience screen, not statistical confidence
}
EVIDENCE_DIR = BASE/'analog_atlas_evidence'
EVIDENCE_DIR.mkdir(exist_ok=True)
OBSERVATIONS_PATH = EVIDENCE_DIR/f'reviewed_{hashlib.sha256(EVIDENCE_CANDIDATE_ID.encode()).hexdigest()[:16]}.json'
if not OBSERVATIONS_PATH.exists():
    OBSERVATIONS_PATH.write_text(json.dumps(template(EVIDENCE_CANDIDATE_ID),indent=2),encoding='utf-8')
OBSERVATIONS = json.loads(OBSERVATIONS_PATH.read_text(encoding='utf-8'))
CATALOGS = {}
if RUN_EVIDENCE_APIS:
    for source_id in SOURCES:
        CATALOGS[source_id] = query_catalog(source_id,EVIDENCE_BBOX,EVIDENCE_START,EVIDENCE_END)
        print(source_id, CATALOGS[source_id]['status'])
    (EVIDENCE_DIR/'catalog_queries.json').write_text(json.dumps(CATALOGS,indent=2),encoding='utf-8')
else:
    print('Live queries off. Switch RUN_EVIDENCE_APIS to True and rerun this cell to check coverage.')
print('Candidate:',EVIDENCE_CANDIDATE_ID)
print('Reviewed observation file:',OBSERVATIONS_PATH)
EVIDENCE_REPORT = {}
def run_evidence_check(name):
    result = assess(name,EVIDENCE_CANDIDATE,OBSERVATIONS.get(name,{}),SETTINGS,CATALOGS)
    EVIDENCE_REPORT[name]=result
    display({k:result[k] for k in ['check','status','finding','satellite_can_support','field_testing_required','catalog_status']})
    return result

In [ ]:
SATELLITE_ENGINE_SOURCE = '"""Bounded Earthdata extraction for EMIT V001, MOD11A1.061, MOD10A1.61.\n\nNative pixel centers are clipped to the candidate polygon. QA decisions and\nsource files are retained. No field review or planetary mineral match is inferred.\n"""\nfrom collections import Counter, defaultdict\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nfrom pathlib import Path\nimport re\nfrom urllib.parse import urlparse\nimport numpy as np\n\nDEFAULTS = dict(max_emit_scenes=3, max_modis_days=12, max_catalog_items=2000,\n                max_thermal_days=60, target_thermal_summaries=24, target_thermal_months=6,\n                max_download_gb=2.0, max_file_mb=300, emit_fit_min=.8,\n                emit_fit_multiplier=2.0, emit_min_depth_sigma=3.0,\n                emit_min_detection_pixels=3, lst_max_error_code=0,\n                lst_max_emissivity_error_code=1, snow_max_basic_qa=1,\n                snow_reject_bits=[0,1,2,3,4,5,7], snow_ndsi_threshold=40,\n                minimum_valid_fraction=.1)\nCMR=\'https://cmr.earthdata.nasa.gov/search/granules.json\'\n\ndef save(path,value):\n    Path(path).write_text(json.dumps(value,indent=2,allow_nan=False),encoding=\'utf-8\')\n\ndef error_detail(error):\n    result={\'type\':type(error).__name__}\n    if isinstance(error,StopIteration):result[\'reason\']=\'Required companion file link missing from the catalog.\'\n    elif isinstance(error,KeyError):result[\'reason\']=\'Required data variable or metadata key is missing.\'\n    elif isinstance(error,ValueError):result[\'reason\']=str(error)[:240]\n    if getattr(error,\'response\',None) is not None:result[\'http_status\']=error.response.status_code\n    return result\n\ndef clean_array(variable, key=slice(None)):\n    return np.ma.asarray(variable[key],dtype=float).filled(np.nan)\n\ndef pixel_roi(lon,lat,polygon):\n    from shapely import contains_xy\n    from shapely.geometry import Polygon\n    return np.isfinite(lon)&np.isfinite(lat)&contains_xy(Polygon(polygon),lon,lat)\n\ndef lst_mask(raw,qc,roi,config):\n    qc=np.asarray(qc,dtype=np.uint16)\n    return (roi & np.isfinite(raw) & (raw>=7500)&(raw<=65535)\n            & ((qc&3)==0)&(((qc>>2)&3)==0)\n            & (((qc>>4)&3)<=config[\'lst_max_emissivity_error_code\'])\n            & (((qc>>6)&3)<=config[\'lst_max_error_code\']))\n\ndef lst_audit(raw,qc,roi,config,fill=None):\n    """Sequential, exclusive pixel losses. QA thresholds are unchanged."""\n    qc=np.asarray(qc,dtype=np.uint16)\n    stages=[(\'valid_raw_range\',np.isfinite(raw)&(raw>=7500)&(raw<=65535)),\n            (\'mandatory_good\',(qc&3)==0),(\'data_quality_good\',((qc>>2)&3)==0),\n            (\'emissivity_error\',((qc>>4)&3)<=config[\'lst_max_emissivity_error_code\']),\n            (\'temperature_error\',((qc>>6)&3)<=config[\'lst_max_error_code\'])]\n    if fill is not None:stages[0]=(stages[0][0],stages[0][1]&(raw!=fill))\n    keep=roi.copy();losses={};remaining={\'roi_pixels\':int(keep.sum())}\n    for name,screen in stages:\n        previous=int(keep.sum());keep &= screen\n        remaining[name]=int(keep.sum());losses[name]=previous-int(keep.sum())\n    return keep,{\'remaining\':remaining,\'exclusive_losses\':losses,\n                 \'mandatory_qc_codes_in_roi\':{str(v):int(np.sum(roi&((qc&3)==v))) for v in range(4)}}\n\ndef thermal_date_order(entries,initial,maximum):\n    """Keep the original evenly spaced dates, then fill the largest date gaps."""\n    dates=sorted({e[\'time_start\'][:10] for e in entries})\n    if not dates:return []\n    selected=list(dict.fromkeys(dates[i] for i in np.linspace(0,len(dates)-1,min(initial,len(dates)),dtype=int)))\n    day=lambda s:datetime.fromisoformat(s).toordinal()\n    while len(selected)<min(maximum,len(dates)):\n        remaining=[d for d in dates if d not in selected]\n        best=max(remaining,key=lambda d:(min(abs(day(d)-day(s)) for s in selected),-day(d)))\n        selected.append(best)\n    return selected[:maximum]\n\ndef summary_reason(roi,valid,fraction,incomplete,minimum):\n    if incomplete:return \'incomplete_tiles\'\n    if not roi:return \'no_pixels_inside_polygon\'\n    if not valid:return \'no_pixels_passed_QA\'\n    if fraction<minimum:return \'below_valid_fraction\'\n    return \'accepted\'\n\ndef coverage_diagnosis(detail,minimum_valid_fraction=.1):\n    rows=detail.get(\'observations\',[])\n    reasons=Counter(r.get(\'rejection_reason\') or (\'accepted\' if r.get(\'accepted\') else summary_reason(r.get(\'roi_pixels\',0),r.get(\'valid_pixels\',0),r.get(\'valid_fraction\',0),r.get(\'incomplete_tiles\',False),minimum_valid_fraction)) for r in rows)\n    losses=Counter()\n    for row in rows:losses.update(row.get(\'qa_exclusive_losses\',{}))\n    accepted=[r for r in rows if r.get(\'accepted\')]\n    return {\'accepted_summaries\':len(accepted),\'accepted_days\':len({r.get(\'date\') for r in accepted}),\n            \'accepted_months\':sorted({r[\'date\'][:7] for r in accepted if r.get(\'date\')}),\n            \'summary_outcomes\':dict(reasons),\'exclusive_pixel_QA_losses\':dict(losses),\n            \'failed_granules\':len(detail.get(\'errors\',[])),\n            \'note\':\'Older manifests lack pixel rejection counts. Missing diagnostics cannot establish why old pixels were lost.\'}\n\ndef snow_mask(ndsi,basic,flags,roi,config):\n    reject=sum(1<<b for b in set(config[\'snow_reject_bits\']))\n    return (roi & (ndsi>=0)&(ndsi<=100)&(basic<=config[\'snow_max_basic_qa\'])\n            & ((flags.astype(np.uint16)&reject)==0))\n\ndef emit_detection_mask(ids,depth,uncertainty,fit,clear,config):\n    fit=fit*config[\'emit_fit_multiplier\']\n    return (clear & np.isfinite(ids)&(ids>0)&np.isfinite(depth)&(depth>0)\n            & np.isfinite(uncertainty)&(uncertainty>0)\n            & (depth>=config[\'emit_min_depth_sigma\']*uncertainty)\n            & np.isfinite(fit)&(fit>=config[\'emit_fit_min\'])&(fit<=1.00001))\n\ndef catalog(short_name,version,bbox,start,end,limit,producer=None):\n    import requests\n    params={\'short_name\':short_name,\'version\':version,\'page_size\':limit,\'sort_key\':\'start_date\'}\n    if producer:params[\'granule_ur\']=producer\n    else:params.update(bounding_box=\',\'.join(map(str,bbox)),temporal=f\'{start}T00:00:00Z,{end}T23:59:59Z\')\n    response=requests.get(CMR,params=params,timeout=(15,60));response.raise_for_status()\n    entries=response.json()[\'feed\'][\'entry\'];hits=int(response.headers.get(\'CMR-Hits\',len(entries)))\n    if hits>limit:raise ValueError(\'Catalog exceeds configured cap; shorten the date range or increase max_catalog_items.\')\n    return entries\n\ndef data_urls(entry):\n    return sorted(set(l[\'href\'] for l in entry.get(\'links\',[]) if not l.get(\'inherited\')\n        and l.get(\'rel\',\'\').endswith(\'/data#\') and l.get(\'href\',\'\').startswith(\'https://\')\n        and urlparse(l[\'href\']).path.lower().endswith((\'.nc\',\'.hdf\'))))\n\ndef select_days(entries,maximum):\n    """Spread observations across the available dates; retain every tile that day."""\n    dates=sorted(set(e[\'time_start\'][:10] for e in entries))\n    selected={dates[i] for i in np.unique(np.linspace(0,len(dates)-1,min(maximum,len(dates)),dtype=int))} if dates else set()\n    # Deduplicate archive copies by product/date/tile; use latest processing name.\n    unique={}\n    for entry in entries:\n        if entry[\'time_start\'][:10] not in selected:continue\n        for url in data_urls(entry):\n            name=Path(urlparse(url).path).name\n            if not name.endswith(\'.hdf\'):continue\n            key=\'.\'.join(name.split(\'.\')[:4])\n            if key not in unique or name>unique[key][0]:unique[key]=(name,entry,url)\n    return [(v[1],v[2]) for _,v in sorted(unique.items())]\n\nclass Downloader:\n    def __init__(self,session,directory,config):\n        self.session=session;self.directory=Path(directory);self.directory.mkdir(parents=True,exist_ok=True)\n        self.config=config;self.bytes=0;self.assets={}\n        if hasattr(session,\'mount\'):\n            from requests.adapters import HTTPAdapter\n            from urllib3.util.retry import Retry\n            session.mount(\'https://\',HTTPAdapter(max_retries=Retry(total=2,backoff_factor=1,\n                status_forcelist=[429,500,502,503,504],allowed_methods=[\'GET\'],respect_retry_after_header=True)))\n    def fetch(self,url):\n        parsed=urlparse(url)\n        if parsed.scheme!=\'https\' or not (parsed.hostname.endswith(\'.nasa.gov\') or parsed.hostname.endswith(\'.nsidc.org\') or parsed.hostname==\'nsidc.org\'):\n            raise ValueError(\'Unexpected Earthdata download host\')\n        name=Path(parsed.path).name\n        path=self.directory/name\n        cap=int(self.config[\'max_file_mb\']*1e6)\n        if not path.exists():\n            part=path.with_suffix(path.suffix+\'.part\')\n            try:\n                with self.session.get(url,stream=True,timeout=(20,90)) as response:\n                    response.raise_for_status()\n                    if \'text/html\' in response.headers.get(\'Content-Type\',\'\'):raise ValueError(\'Login page returned instead of data\')\n                    expected=int(response.headers.get(\'Content-Length\',0))\n                    if expected>cap:raise ValueError(\'File exceeds max_file_mb\')\n                    size=0\n                    with part.open(\'wb\') as stream:\n                        for block in response.iter_content(1024*1024):\n                            size+=len(block);self.bytes+=len(block)\n                            if size>cap or self.bytes>self.config[\'max_download_gb\']*1e9:raise ValueError(\'Download budget exceeded\')\n                            stream.write(block)\n                    if expected and size!=expected:raise ValueError(\'Incomplete download\')\n                    part.replace(path)\n            finally:\n                if part.exists():part.unlink()\n        # Reject HTML/error cache entries; HDF4 or HDF5/NetCDF4 signatures only.\n        with path.open(\'rb\') as stream:\n            signature=stream.read(8)\n            if not (signature.startswith(b\'\\x0e\\x03\\x13\\x01\') or signature==b\'\\x89HDF\\r\\n\\x1a\\n\'):\n                raise ValueError(\'Cached file is not HDF4/NetCDF4; remove this cached file and retry\')\n            stream.seek(0);digest=hashlib.file_digest(stream,\'sha256\').hexdigest()\n        self.assets[url]={\'url\':url,\'file\':name,\'bytes\':path.stat().st_size,\'sha256\':digest}\n        return path\n\ndef modis_window(sd,shape,polygon):\n    """Read sinusoidal HDF-EOS bounds, then mask exact geographic pixel centers."""\n    from pyproj import CRS,Transformer\n    attrs=sd.attributes();meta=\'\'.join(str(attrs[k]) for k in sorted(attrs) if k.startswith(\'StructMetadata.\'))\n    if \'GCTP_SNSOID\' not in meta:raise ValueError(\'Expected MODIS sinusoidal grid\')\n    def pair(name):\n        match=re.search(name+r\'\\s*=\\s*\\(([^)]+)\\)\',meta)\n        if not match:raise ValueError(\'Missing HDF-EOS grid bounds\')\n        return [float(v) for v in match[1].split(\',\')]\n    left,top=pair(\'UpperLeftPointMtrs\');right,bottom=pair(\'LowerRightMtrs\')\n    params=pair(\'ProjParams\');radius=params[0] if params[0]>0 else 6371007.181\n    crs=CRS.from_proj4(f\'+proj=sinu +R={radius} +units=m +no_defs\')\n    forward=Transformer.from_crs(\'EPSG:4326\',crs,always_xy=True)\n    inverse=Transformer.from_crs(crs,\'EPSG:4326\',always_xy=True)\n    edge=[]\n    for a,b in zip(polygon[:-1],polygon[1:]):\n        edge.extend([(a[0]+t*(b[0]-a[0]),a[1]+t*(b[1]-a[1])) for t in np.linspace(0,1,33)])\n    x,y=forward.transform(*np.array(edge).T)\n    height,width=shape;dx=(right-left)/width;dy=(top-bottom)/height\n    c0=max(0,int(math.floor((min(x)-left)/dx)));c1=min(width,int(math.ceil((max(x)-left)/dx)))\n    r0=max(0,int(math.floor((top-max(y))/dy)));r1=min(height,int(math.ceil((top-min(y))/dy)))\n    if c1<=c0 or r1<=r0:return None\n    xx,yy=np.meshgrid(left+(np.arange(c0,c1)+.5)*dx,top-(np.arange(r0,r1)+.5)*dy)\n    lon,lat=inverse.transform(xx,yy)\n    return (slice(r0,r1),slice(c0,c1)),pixel_roi(lon,lat,polygon)\n\ndef extract_modis(path,kind,polygon,config):\n    from pyhdf.SD import SD,SDC\n    sd=SD(str(path),SDC.READ)\n    try:\n        first=sd.select(\'LST_Day_1km\' if kind==\'thermal\' else \'NDSI_Snow_Cover\')\n        shape=first.info()[2];first.endaccess()\n        window=modis_window(sd,shape,polygon)\n        if window is None:return []\n        slices,roi=window\n        def get(name):\n            variable=sd.select(name)\n            try:return np.array(variable[slices]),variable.attributes()\n            finally:variable.endaccess()\n        results=[]\n        if kind==\'thermal\':\n            for period in [\'Day\',\'Night\']:\n                raw,attrs=get(f\'LST_{period}_1km\');qc,_=get(f\'QC_{period}\')\n                valid,audit=lst_audit(raw,qc,roi,config,attrs.get(\'_FillValue\'))\n                scale=float(attrs.get(\'scale_factor\',.02));offset=float(attrs.get(\'add_offset\',0))\n                if not np.isclose(scale,.02) or offset!=0:raise ValueError(\'Unexpected MOD11 scale/offset\')\n                values=(raw[valid].astype(float)*scale-273.15).tolist()\n                results.append({\'period\':period,\'roi_pixels\':int(roi.sum()),\'valid_pixels\':int(valid.sum()),\'values_c\':values,\'qa_audit\':audit})\n        else:\n            ndsi,_=get(\'NDSI_Snow_Cover\');basic,_=get(\'NDSI_Snow_Cover_Basic_QA\');flags,_=get(\'NDSI_Snow_Cover_Algorithm_Flags_QA\')\n            valid=snow_mask(ndsi,basic,flags,roi,config)\n            results.append({\'period\':\'Day\',\'roi_pixels\':int(roi.sum()),\'valid_pixels\':int(valid.sum()),\n                            \'snow_pixels\':int(np.sum(valid&(ndsi>=config[\'snow_ndsi_threshold\'])))})\n        return results\n    finally:sd.end()\n\ndef extract_emit(min_path,unc_path,mask_path,polygon,config):\n    from netCDF4 import Dataset\n    with Dataset(min_path) as minerals,Dataset(unc_path) as uncertainty,Dataset(mask_path) as masks:\n        loc=minerals.groups[\'location\'];lat=clean_array(loc[\'lat\']);lon=clean_array(loc[\'lon\'])\n        roi=pixel_roi(lon,lat,polygon)\n        # Companion files must represent identical native detector geometry.\n        for dataset in [uncertainty,masks]:\n            other=dataset.groups[\'location\']\n            if other[\'lat\'].shape!=lat.shape or not np.allclose(clean_array(other[\'lat\'])[roi],lat[roi],atol=1e-6,equal_nan=True) or not np.allclose(clean_array(other[\'lon\'])[roi],lon[roi],atol=1e-6,equal_nan=True):\n                raise ValueError(\'EMIT companion geolocation mismatch\')\n        labels=[str(v).lower() for v in masks.groups[\'sensor_band_parameters\'][\'mask_bands\'][:]]\n        if len(labels)<8 or not all(token in labels[i] for i,token in enumerate([\'cloud\',\'cirrus\',\'water\',\'spacecraft\',\'dilated\'])):\n            raise ValueError(\'Unsupported EMIT mask layout; V001 required\')\n        clear=roi.copy()\n        # Flag bands only; aerosol/water-vapor values at 5/6 are not binary flags.\n        for index in [0,1,2,3,4,7]:clear &= clean_array(masks[\'mask\'],(slice(None),slice(None),index))==0\n        metadata=minerals.groups[\'mineral_metadata\']\n        names={int(i):str(n) for i,n in zip(metadata[\'index\'][:],metadata[\'name\'][:])}\n        counts=Counter();group_stats=[]\n        for group in [1,2]:\n            prefix=f\'group_{group}_\'\n            ids=clean_array(minerals[prefix+\'mineral_id\']);depth=clean_array(minerals[prefix+\'band_depth\'])\n            unc=clean_array(uncertainty[prefix+\'band_depth_unc\']);fit=clean_array(uncertainty[prefix+\'fit\'])\n            accepted=emit_detection_mask(ids,depth,unc,fit,clear,config)\n            found,cnt=np.unique(ids[accepted].astype(int),return_counts=True)\n            for ident,count in zip(found,cnt):\n                if ident not in names:raise ValueError(\'Mineral ID absent from file metadata\')\n                counts[names[ident]]+=int(count)\n            group_stats.append({\'group\':group,\'accepted_detection_pixels\':int(accepted.sum())})\n        return {\'roi_pixels\':int(roi.sum()),\'clear_pixels\':int(clear.sum()),\'counts\':dict(counts),\'groups\':group_stats}\n\ndef run_extraction(candidate,start,end,session,directory,config=None):\n    cfg={**DEFAULTS,**(config or {})}\n    if cfg[\'max_emit_scenes\']<1 or cfg[\'max_modis_days\']<1:raise ValueError(\'Download limits must be positive\')\n    if not 0<cfg[\'minimum_valid_fraction\']<=1:raise ValueError(\'minimum_valid_fraction must be in (0,1]\')\n    polygon=candidate[\'footprint\'];box=[min(p[0] for p in polygon),min(p[1] for p in polygon),max(p[0] for p in polygon),max(p[1] for p in polygon)]\n    if box[2]-box[0]>180:raise ValueError(\'Split date-line-crossing polygons before extraction\')\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    previous=None\n    previous_path=directory/\'extraction_manifest.json\'\n    if previous_path.exists():\n        old=json.loads(previous_path.read_text(encoding=\'utf-8\'))\n        if old.get(\'candidate_id\')==candidate[\'id\'] and old.get(\'period\')==[start,end]:\n            previous=coverage_diagnosis(old.get(\'products\',{}).get(\'thermal\',{}),old.get(\'settings\',{}).get(\'minimum_valid_fraction\',.1))\n            save(directory/\'previous_extraction_manifest.json\',old)\n    dl=Downloader(session,directory/\'downloads\',cfg)\n    report={\'candidate_id\':candidate[\'id\'],\'period\':[start,end],\'settings\':cfg,\n            \'created_utc\':datetime.now(timezone.utc).isoformat(),\'products\':{},\'assets\':[]}\n    report[\'packages\']={name:importlib.metadata.version(name) for name in [\'numpy\',\'netCDF4\',\'pyhdf\',\'pyproj\',\'shapely\']}\n    report[\'selection_note\']=\'EMIT scenes and MODIS dates sampled across available catalog results; all returned MODIS tiles retained for each selected date. No claim of full-year or full-patch valid coverage.\'\n    report[\'previous_thermal_diagnosis\']=previous\n    observations={}\n    def base(method,urls):\n        return dict(candidate_id=candidate[\'id\'],source_urls=sorted(set(urls)),period=f\'{start}/{end}\',method=method,\n                    qa_reviewed=False,qa_processed=True,processing=\'satellite_extract_v1\',field_validated=False)\n    for kind,product,version in [(\'thermal\',\'MOD11A1\',\'061\'),(\'ice\',\'MOD10A1\',\'61\'),(\'minerals\',\'EMITL2BMIN\',\'001\')]:\n        print(f\'Extracting {product}...\',flush=True)\n        detail={\'status\':\'STARTED\',\'errors\':[],\'observations\':[]};report[\'products\'][kind]=detail\n        used_urls=[];values=[];fractions=[];mineral_counts=Counter();groups=defaultdict(list)\n        try:\n            entries=catalog(product,version,box,start,end,cfg[\'max_catalog_items\'])\n            detail[\'catalog_count\']=len(entries)\n            if kind==\'minerals\':\n                picks=np.unique(np.linspace(0,len(entries)-1,min(cfg[\'max_emit_scenes\'],len(entries)),dtype=int)) if entries else []\n                detail[\'selected_granules\']=len(picks)\n                for index in picks:\n                    entry=entries[index]\n                    try:\n                        urls=data_urls(entry)\n                        min_url=next(u for u in urls if Path(urlparse(u).path).name.startswith(\'EMIT_L2B_MIN_001_\'))\n                        unc_url=next(u for u in urls if Path(urlparse(u).path).name.startswith(\'EMIT_L2B_MINUNCERT_001_\'))\n                        suffix=Path(urlparse(min_url).path).name.split(\'EMIT_L2B_MIN_001_\')[1].removesuffix(\'.nc\')\n                        companion=catalog(\'EMITL2ARFL\',\'001\',box,start,end,50,\'EMIT_L2A_RFL_001_\'+suffix)\n                        mask_url=next(u for e in companion for u in data_urls(e) if Path(urlparse(u).path).name==\'EMIT_L2A_MASK_001_\'+suffix+\'.nc\')\n                        paths=[dl.fetch(u) for u in [min_url,unc_url,mask_url]]\n                        result=extract_emit(*paths,polygon,cfg)\n                        result.update(granule=entry.get(\'title\',entry[\'id\']),time=entry[\'time_start\'])\n                        result[\'accepted_scene\']=result[\'roi_pixels\']>0 and result[\'clear_pixels\']/max(1,result[\'roi_pixels\'])>=cfg[\'minimum_valid_fraction\']\n                        detail[\'observations\'].append(result)\n                        if result[\'accepted_scene\']:\n                            mineral_counts.update(result[\'counts\']);used_urls.extend([min_url,unc_url,mask_url])\n                    except Exception as error:\n                        detail[\'errors\'].append({\'granule\':entry[\'id\'],**error_detail(error),\'stage\':\'EMIT companion/download/extraction\'})\n                names=sorted(n for n,c in mineral_counts.items() if c>=cfg[\'emit_min_detection_pixels\'])\n                record=base(\'Native EMIT pixel centers clipped to polygon; V001 mask flags 0,1,2,3,4,7 clear; positive detections; band depth >= configured sigma * uncertainty; fit * 2 >= configured minimum. Counts are pixel-detections, not mineral abundance.\',used_urls)\n                record.update(earth_minerals=names,target_minerals=[],cross_sensor_reviewed=False,mineral_pixel_detections=dict(mineral_counts))\n                has_values=bool(names)\n            else:\n                selected=select_days(entries,len(entries) or 1) if kind==\'thermal\' else select_days(entries,cfg[\'max_modis_days\'])\n                if kind==\'thermal\':\n                    order=thermal_date_order(entries,cfg[\'max_modis_days\'],cfg[\'max_thermal_days\'])\n                    selected=sorted([(e,u) for e,u in selected if e[\'time_start\'][:10] in order],key=lambda pair:order.index(pair[0][\'time_start\'][:10]))\n                    detail[\'planned_dates\']=order\n                detail[\'selected_granules\']=len(selected)\n                failed_dates=set()\n                attempted_dates=[];accepted_so_far=[];stop_reason=\'date_limit_or_available_dates\'\n                for position,(entry,url) in enumerate(selected):\n                    date=entry[\'time_start\'][:10]\n                    if date not in attempted_dates:\n                        if kind==\'thermal\' and len(attempted_dates)>=cfg[\'max_modis_days\']:\n                            accepted_so_far=[key for key,parts in groups.items() if summary_reason(\n                                sum(p[\'roi_pixels\'] for p,u in parts),sum(p[\'valid_pixels\'] for p,u in parts),\n                                sum(p[\'valid_pixels\'] for p,u in parts)/max(1,sum(p[\'roi_pixels\'] for p,u in parts)),\n                                key[0] in failed_dates,cfg[\'minimum_valid_fraction\'])==\'accepted\']\n                            if len(accepted_so_far)>=cfg[\'target_thermal_summaries\'] and len({key[0][:7] for key in accepted_so_far})>=cfg[\'target_thermal_months\']:\n                                stop_reason=\'coverage_targets_met\';break\n                        if dl.bytes>=cfg[\'max_download_gb\']*1e9:\n                            stop_reason=\'download_budget\';break\n                        attempted_dates.append(date)\n                    try:\n                        path=dl.fetch(url)\n                        parts=extract_modis(path,\'thermal\' if kind==\'thermal\' else \'ice\',polygon,cfg)\n                        for part in parts:\n                            key=(entry[\'time_start\'][:10],part[\'period\']);groups[key].append((part,url))\n                    except Exception as error:\n                        failed_dates.add(entry[\'time_start\'][:10])\n                        detail[\'errors\'].append({\'granule\':entry[\'id\'],**error_detail(error),\'stage\':\'MODIS download/extraction\'})\n                        detail[\'errors\'][-1][\'date\']=date\n                detail[\'attempted_dates\']=attempted_dates;detail[\'stop_reason\']=stop_reason\n                for date in attempted_dates:\n                    for period in ([\'Day\',\'Night\'] if kind==\'thermal\' else [\'Day\']):groups.setdefault((date,period),[])\n                for (date,period),parts in sorted(groups.items()):\n                    roi=sum(p[\'roi_pixels\'] for p,u in parts);valid=sum(p[\'valid_pixels\'] for p,u in parts)\n                    item={\'date\':date,\'period\':period,\'roi_pixels\':roi,\'valid_pixels\':valid,\'valid_fraction\':valid/max(1,roi)}\n                    item[\'incomplete_tiles\']=date in failed_dates\n                    item[\'accepted\']=valid>0 and not item[\'incomplete_tiles\'] and item[\'valid_fraction\']>=cfg[\'minimum_valid_fraction\']\n                    item[\'rejection_reason\']=summary_reason(roi,valid,item[\'valid_fraction\'],item[\'incomplete_tiles\'],cfg[\'minimum_valid_fraction\'])\n                    if kind==\'thermal\':\n                        qa_losses=Counter()\n                        for p,u in parts:qa_losses.update(p.get(\'qa_audit\',{}).get(\'exclusive_losses\',{}))\n                        item[\'qa_exclusive_losses\']=dict(qa_losses)\n                        local=[v for p,u in parts for v in p[\'values_c\']]\n                        if local:item.update(min_c=min(local),max_c=max(local),mean_c=float(np.mean(local)))\n                        if item[\'accepted\']:values.extend(local)\n                    else:\n                        snow=sum(p[\'snow_pixels\'] for p,u in parts)\n                        item[\'snow_fraction_of_valid_pixels\']=snow/valid if valid else None\n                        if item[\'accepted\']:fractions.append(snow/valid)\n                    if item[\'accepted\']:used_urls.extend(u for p,u in parts)\n                    detail[\'observations\'].append(item)\n                if kind==\'thermal\':\n                    detail[\'diagnosis\']=coverage_diagnosis(detail,cfg[\'minimum_valid_fraction\'])\n                    save(directory/\'temperature_coverage.json\',{\'previous\':previous,\'current\':detail[\'diagnosis\'],\n                        \'attempted_dates\':attempted_dates,\'stop_reason\':stop_reason,\'errors\':detail[\'errors\'],\n                        \'quality_masks_relaxed\':False,\'settings\':cfg})\n                    print(\'Temperature coverage:\',json.dumps(detail[\'diagnosis\']),flush=True)\n                record=base(\'MODIS native sinusoidal pixel centers clipped to polygon; fill, valid-range and product-specific QA applied. Selected dates evenly spaced across available observations; not a full climatology.\',used_urls)\n                if kind==\'thermal\':\n                    record.update(surface_temperature_c=values,sample_unit=\'valid pixel-observations\',\n                                  observation_count=sum(o[\'accepted\'] for o in detail[\'observations\']))\n                    has_values=bool(values)\n                else:\n                    record.update(snow_cover_fraction=fractions,snow_ndsi_threshold=cfg[\'snow_ndsi_threshold\'],\n                                  fraction_definition=\'Fraction of QA-valid observed pixels with NDSI_Snow_Cover >= threshold. NDSI itself is not fractional snow cover.\')\n                    has_values=bool(fractions)\n            if has_values:status=\'AUTO_QA_MEASURED\'\n            elif not entries:status=\'NO_GRANULES\'\n            elif detail[\'errors\'] and detail[\'observations\'] and all(r.get(\'incomplete_tiles\') for r in detail[\'observations\']):status=\'EXTRACTION_FAILED\'\n            elif detail[\'observations\']:status=\'NO_ACCEPTED_MEASUREMENTS\'\n            else:status=\'EXTRACTION_FAILED\'\n            detail[\'status\']=status;record[\'extraction_status\']=status\n            if not has_values:record[\'qa_processed\']=False\n            observations[kind]=record\n        except Exception as error:\n            detail.update(status=\'EXTRACTION_FAILED\',error=error_detail(error))\n            observations[kind]={**base(\'Extraction did not complete.\',[]),\'qa_processed\':False,\'extraction_status\':\'EXTRACTION_FAILED\'}\n        # Preserve partial results if a later product fails.\n        report[\'assets\']=list(dl.assets.values());save(directory/\'extraction_manifest.json\',report)\n        save(directory/\'automatic_observations.json\',observations)\n        print(kind+\': \'+detail[\'status\'],flush=True)\n    observations[\'construction\']=dict(observations[\'minerals\'])\n    save(directory/\'automatic_observations.json\',observations)\n    return observations,report\n'

In [ ]:
RUN_SATELLITE_EXTRACTION = True
lower_latitude_ids = [r['id'] for r in RANKINGS[TARGET] if r['id'] in DISCOVERY_IDS and abs(r['lat'])<50][:1]
EXTRACTION_CANDIDATE_IDS = list(dict.fromkeys([EVIDENCE_CANDIDATE_ID]+lower_latitude_ids+DISCOVERY_IDS))[:SATELLITE_CANDIDATE_COUNT]
# Optional batch: add other DATA row IDs to this list; cap applies per candidate.
# The download cap below applies PER candidate. Keep the batch small.
EXTRACTION_CONFIG = {
    'max_emit_scenes': 3,
    'max_modis_days': 12,
    'max_thermal_days': 60,          # More dates, unchanged pixel-quality masks
    'target_thermal_summaries': 24,  # Accepted day/night summaries; not a confidence claim
    'target_thermal_months': 6,      # Calendar-month coverage goal
    'max_download_gb': 2.0,
    'emit_fit_min': 0.8,
    'emit_fit_multiplier': 2.0,  # V001 fit correction documented by NASA; do not mix versions
    'emit_min_depth_sigma': 3.0,
    'emit_min_detection_pixels': 3,
    'lst_max_error_code': 0,    # QC code 0: estimated LST error <=1 K
    'lst_max_emissivity_error_code': 1,
    'snow_max_basic_qa': 1,     # Best/good
    'snow_reject_bits': [0,1,2,3,4,5,7],
    'snow_ndsi_threshold': 40,  # Index 0..100, not a percent snow value
    'minimum_valid_fraction': 0.10,
}

def perform_satellite_extraction():
    import os, sys, subprocess, types, hashlib
    credentials = {}
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        credentials = {k:os.environ.get(k,'') for k in ['EARTHDATA_USERNAME','EARTHDATA_PASSWORD']}
    else:
        try:
            client=UserSecretsClient()
            credentials={k:client.get_secret(k) for k in ['EARTHDATA_USERNAME','EARTHDATA_PASSWORD']}
        except Exception:
            credentials={}
    if not all(credentials.get(k) for k in ['EARTHDATA_USERNAME','EARTHDATA_PASSWORD']):
        print('EARTHDATA_LOGIN_REQUIRED: add and enable both Earthdata secrets, then rerun this cell.')
        return None, 'EARTHDATA_LOGIN_REQUIRED'
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
        'earthaccess==0.19.0','netCDF4>=1.7,<2','pyhdf==0.11.7','pyproj>=3.6,<4','shapely>=2,<3'])
    import earthaccess
    previous={k:os.environ.get(k) for k in credentials}
    # A pre-existing token must not silently override the requested username login.
    previous_token=os.environ.pop('EARTHDATA_TOKEN',None)
    try:
        os.environ.update(credentials)
        auth=earthaccess.login(strategy='environment',persist=False)
        if not auth.authenticated:
            print('EARTHDATA_LOGIN_FAILED: verify the enabled credentials and Earthdata account access.')
            return None, 'EARTHDATA_LOGIN_FAILED'
    except Exception:
        print('EARTHDATA_LOGIN_FAILED: verify credentials, Internet and any pending Earthdata application authorization.')
        return None, 'EARTHDATA_LOGIN_FAILED'
    finally:
        for k,v in previous.items():
            if v is None:os.environ.pop(k,None)
            else:os.environ[k]=v
        if previous_token is not None:os.environ['EARTHDATA_TOKEN']=previous_token
        credentials.clear()
    module=types.ModuleType('satellite_extract')
    exec(compile(SATELLITE_ENGINE_SOURCE,'satellite_extract.py','exec'),module.__dict__)
    engine_path=EVIDENCE_DIR/'satellite_extract.py'
    engine_path.write_text(SATELLITE_ENGINE_SOURCE,encoding='utf-8')
    selected_observations=None
    for candidate_id in dict.fromkeys(EXTRACTION_CANDIDATE_IDS):
        candidate=next(r for r in DATA['rows'] if r['id']==candidate_id)
        folder=EVIDENCE_DIR/'candidates'/hashlib.sha256(candidate_id.encode()).hexdigest()[:16]/'satellite'
        observations,manifest=module.run_extraction(candidate,EVIDENCE_START,EVIDENCE_END,
            earthaccess.get_requests_https_session(),folder,EXTRACTION_CONFIG)
        print('Saved measurements:',candidate_id,folder)
        if candidate_id==EVIDENCE_CANDIDATE_ID:selected_observations=observations
    return selected_observations, 'NOT_IN_EXTRACTION_BATCH' if selected_observations is None else 'COMPLETED'

if RUN_SATELLITE_EXTRACTION:
    automatic, extraction_state = perform_satellite_extraction()
    if automatic is not None:
        OBSERVATIONS.update(automatic)
    else:
        for name in ['minerals','thermal','ice','construction']:
            OBSERVATIONS[name]={'candidate_id':EVIDENCE_CANDIDATE_ID,'extraction_status':extraction_state}
else:
    print('Satellite extraction disabled. Existing manual observations remain in use.')

In [ ]:
for preview_id in EXTRACTION_CANDIDATE_IDS:
    print('\nCANDIDATE:',DATA['names'].get(preview_id,preview_id))
    preview_obs,preview_manifest,preview_note=candidate_visuals.load_measurements(EVIDENCE_DIR,preview_id,[EVIDENCE_START,EVIDENCE_END])
    print(preview_note)
    mineral=preview_obs.get('minerals',{})
    mineral_df=pd.DataFrame([{'library_label':label,'accepted_pixel_detections':mineral.get('mineral_pixel_detections',{}).get(label)}
        for label in mineral.get('earth_minerals',[])],columns=['library_label','accepted_pixel_detections'])
    temperature_pixels_df=pd.DataFrame({'surface_temperature_c':preview_obs.get('thermal',{}).get('surface_temperature_c',[])})
    temperature_daily_df=pd.DataFrame(preview_manifest.get('products',{}).get('thermal',{}).get('observations',[]))
    snow_df=pd.DataFrame(preview_manifest.get('products',{}).get('ice',{}).get('observations',[]))
    source_files_df=pd.DataFrame(preview_manifest.get('assets',[]))
    for name,frame in [('mineral_df',mineral_df),('temperature_pixels_df',temperature_pixels_df),
        ('temperature_daily_df',temperature_daily_df),('snow_df',snow_df),('source_files_df',source_files_df)]:show_head(name,frame)

In [ ]:
import hashlib
import pandas as pd
candidate_run_dir = EVIDENCE_DIR/'candidates'/hashlib.sha256(EVIDENCE_CANDIDATE_ID.encode()).hexdigest()[:16]/'satellite'
coverage_file = candidate_run_dir/'temperature_coverage.json'
coverage_report = None
if coverage_file.exists():
    coverage_report = json.loads(coverage_file.read_text(encoding='utf-8'))
    print('Previous comparable run:', coverage_report.get('previous'))
    print('Current coverage:', coverage_report['current'])
    print('Stopped because:', coverage_report['stop_reason'])
    display(pd.DataFrame(list(coverage_report['current']['summary_outcomes'].items()),columns=['summary outcome','count']))
    display(pd.DataFrame(list(coverage_report['current']['exclusive_pixel_QA_losses'].items()),columns=['pixel rejection stage','count']))
    if coverage_report['errors']:
        display(pd.DataFrame(coverage_report['errors']))
    print('Full diagnostics:', coverage_file)
else:
    print('No new coverage report yet. Run extraction with enabled Earthdata Secrets.')
    print('No coverage diagnosis is available until extraction runs for this candidate.')

### What the satellites support—and what remains untested

| Check | Satellite evidence | Still needs testing |
|---|---|---|
| Minerals | QA-screened Earth spectral-library matches | Comparable planetary spectra; field spectroscopy/XRD |
| Temperature | Sampled surface-temperature range and coverage | Continuous exposure and equipment thermal cycles |
| Ice | Surface snow-index observations | Buried ice depth, fraction and extraction yield |
| Caves | No automatic cave verification | Entrance/interior survey, stability and radio links |
| Mobility | Regional terrain resemblance | Wheel-scale obstacles, slip and traction |
| Construction | Surface feedstock and terrain leads | Chemistry, strength, excavation and energy/yield |

The charts and popup cards show missing evidence explicitly. Only terrain is compared directly with both planetary targets.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

MAP_CANDIDATE_IDS = DISCOVERY_IDS  # Edit this list to show other sampled patches
EXPORT_ALL_CANDIDATE_CHARTS = True
VISUAL_DIR = EVIDENCE_DIR/'visuals'
VISUAL_DIR.mkdir(exist_ok=True)

# Recompute both planetary comparisons with CURRENT notebook weights and grid.
planet_scores = {}
for planet_id,label in [('moon_south_pole','Moon'),('mars_jezero','Mars')]:
    planet_scores[label] = {r['id']:r['terrain_index'] for r in
        rank_candidates(planet_id,GRID_M,WEIGHTS,False,MIN_OPEN_FRACTION)}

candidate_summaries=[]
for candidate_id in dict.fromkeys(MAP_CANDIDATE_IDS):
    candidate=next(r for r in DATA['rows'] if r['id']==candidate_id)
    observations,manifest,note=candidate_visuals.load_measurements(
        EVIDENCE_DIR,candidate_id,[EVIDENCE_START,EVIDENCE_END])
    scores={planet:values[candidate_id] for planet,values in planet_scores.items() if candidate_id in values}
    summary=candidate_visuals.summarize(candidate,observations,manifest,scores,
        [EVIDENCE_START,EVIDENCE_END],DATA['names'].get(candidate_id),note)
    summary['terrain_grid_m']=int(GRID_M)
    summary['note'] += ' Research: '+review_by_id.get(candidate_id,{}).get('status','NOT_SCREENED')+'; novelty not confirmed.'
    candidate_summaries.append(summary)
    if EXPORT_ALL_CANDIDATE_CHARTS:
        name=candidate_visuals.candidate_folder(EVIDENCE_DIR,candidate_id).name
        fig=candidate_visuals.draw_charts(summary,VISUAL_DIR/f'{name}_evidence.png')
        plt.close(fig)

(VISUAL_DIR/'candidate_evidence.json').write_text(json.dumps({
    'grid_m':GRID_M,'weights':WEIGHTS,'date_range':[EVIDENCE_START,EVIDENCE_END],
    'candidates':candidate_summaries},indent=2),encoding='utf-8')

def show_candidate(candidate_id):
    summary=next(s for s in candidate_summaries if s['id']==candidate_id)
    fig=candidate_visuals.draw_charts(summary)
    display(fig)
    plt.close(fig)
    print(summary['note'])
    print('Field tests: mineral samples/XRD; thermal logging; ice cores/radar; traction/strength; cave survey.')

if candidate_summaries:
    try:
        import ipywidgets as widgets
        selector=widgets.Dropdown(options=[(f"{s['name']} ({s['lat']}, {s['lon']})",s['id']) for s in candidate_summaries],
            description='Candidate:',layout=widgets.Layout(width='650px'))
        chart_output=widgets.Output()
        def update_candidate_chart(change=None):
            with chart_output:
                clear_output(wait=True)
                show_candidate(selector.value)
        selector.observe(update_candidate_chart,names='value')
        display(selector,chart_output)
        update_candidate_chart()
    except ImportError:
        show_candidate(candidate_summaries[0]['id'])
    evidence_map=candidate_visuals.build_map(candidate_summaries,VISUAL_DIR/'candidate_evidence_map.html',
                                            None)
    display(evidence_map)
    print('Saved candidate charts, evidence JSON and shareable map:',VISUAL_DIR)
else:
    print('No candidates selected. Edit MAP_CANDIDATE_IDS or relax the ranking filter.')

In [ ]:
# Six checks for every downloaded candidate; no evidence is borrowed across locations.
ALL_EVIDENCE=[]
for cid in EXTRACTION_CANDIDATE_IDS:
    candidate=next(r for r in DATA['rows'] if r['id']==cid)
    obs,manifest,note=candidate_visuals.load_measurements(EVIDENCE_DIR,cid,[EVIDENCE_START,EVIDENCE_END])
    if cid==EVIDENCE_CANDIDATE_ID:obs={**obs,**OBSERVATIONS}
    checks={name:assess(name,candidate,obs.get(name,{}),SETTINGS,CATALOGS) for name in
        ['minerals','thermal','ice','caves','mobility','construction']}
    ALL_EVIDENCE.append({'candidate_id':cid,'place':DATA['names'].get(cid,cid),'checks':checks})
    display(pd.DataFrame([{'place':DATA['names'].get(cid,cid),'check':name,'status':value['status'],
        'finding':value['finding']} for name,value in checks.items()]))
review.dump(EVIDENCE_DIR/'six_evidence_checks_all_candidates.json',ALL_EVIDENCE)

### Save and share

Keep `plain_results.json`, `publication_reviews.json`, `literature_screen.json`, `scrub_audit.json`, the fresh manifests and the evidence folder. They document **where the analogy works, where it breaks, and what to test next**. Re-run with another seed and compare which kinds of terrain keep appearing.

**24-hour plan:** 0–4 h obtain/scrub → 4–8 h inspect/rank → 8–14 h research review → 14–20 h satellite follow-up → 20–24 h explain and share. Network delays and data gaps may change this schedule.

Sources: [Copernicus DEM](https://registry.opendata.aws/copernicus-dem/), [LOLA](https://pds-geosciences.wustl.edu/missions/lro/lola.htm), [MOLA](https://pds-geosciences.wustl.edu/missions/mgs/megdr.html), [WorldCover](https://esa-worldcover.org/en/data-access), [EMIT QA](https://nasa.github.io/LPDAAC-Data-Resources/external/Working_with_EMIT_L2B_Mineralogy.html), [MOD11 QA](https://ladsweb.modaps.eosdis.nasa.gov/filespec/MODIS/61/MOD11A1), [snow QA](https://modis-snow-ice.gsfc.nasa.gov/uploads/snow_user_guide_C6.1_final_revised_april.pdf), [Crossref API](https://www.crossref.org/documentation/retrieve-metadata/rest-api/).

No field access, operational safety or permanent settlement is validated by this notebook. Clear personal information in manual reviews before publishing.

In [ ]:
import importlib.metadata
FINAL_MANIFEST={**DATA['manifest'],'analysis_grid_m':GRID_M,'weights':WEIGHTS,
    'packages':{p:importlib.metadata.version(p) for p in ['numpy','pandas','rasterio','pyproj','shapely','folium']},
    'helper_sha256':{name:hashlib.sha256((HELPER_DIR/name).read_bytes()).hexdigest() for name in BUNDLE},
    'known_registry_sha256':hashlib.sha256(json.dumps(KNOWN,sort_keys=True).encode()).hexdigest(),
    'shortlisted_ids':SHORTLIST_IDS,'publication_reviewed_known_ids':sorted(KNOWN_FROM_PAPERS),
    'claim':'Terrain leads with research screening; no confirmed new analog or permanent-base suitability.'}
review.dump(EXPORT_DIR/'final_run_manifest.json',FINAL_MANIFEST)
print('Results:',EXPORT_DIR)
print('Evidence:',EVIDENCE_DIR)